# MR3a — Daily Final Join (Reduce-Side)
**Inputs**: `mr3/daily_crash_mr.csv` + `mr3/address_store_mr.csv` + `mr3/type_count_store_mr.csv`
**Output**: `mr3/mr_daily_final.csv`

In [1]:
import json
import pandas as pd
from collections import defaultdict

CRASH_INPUT  = 'daily_crash_mr.csv'
ADDR_INPUT   = 'address_store_mr.csv'
TYPE_INPUT   = 'type_count_store_mr.csv'
OUTPUT       = 'mr_daily_final.csv'

crashes = pd.read_csv(CRASH_INPUT)
addr    = pd.read_csv(ADDR_INPUT)
types   = pd.read_csv(TYPE_INPUT)

print(f'Crash rows : {len(crashes):,}  |  zones: {crashes["zone_id"].nunique():,}')
print(f'Store zones: {len(addr):,}')
print(f'Type rows  : {len(types):,}')

Crash rows : 2,891  |  zones: 1,490
Store zones: 1,823
Type rows  : 6,576


In [ ]:
# ── MAP ───────────────────────────────────────────────────────────────────────
type_by_zone = defaultdict(dict)
for _, row in types.iterrows():
    type_by_zone[str(row['zone_id'])][row['method_of_operation']] = int(row['count'])

def mapper(row, source):
    zid = str(row['zone_id'])
    if source == 'crash':
        return zid, {
            'src':        'crash',
            'crash_date': row['crash_date'],
            'crashes':    row['crashes'],
            'injured':    row['injured'],
            'killed':     row['killed'],
        }
    else:
        return zid, {
            'src':               'store',
            'store_count':       row['store_count'],
            'active_licenses':   row['active_licenses'],
            'outdated_licenses': row['outdated_licenses'],
            'type_counts_json':  json.dumps(type_by_zone.get(zid, {})),
        }

mapped  = [mapper(row, 'crash') for _, row in crashes.iterrows()]
mapped += [mapper(row, 'store') for _, row in addr.iterrows()]

print(f'Total mapped pairs: {len(mapped):,}')

Total mapped pairs: 4,714


In [ ]:
# ── SHUFFLE ───────────────────────────────────────────────────────────────────
shuffled = defaultdict(list)
for zone_id, record in mapped:
    shuffled[zone_id].append(record)

print(f'Unique zone_ids after shuffle: {len(shuffled):,}')

Unique zone_ids after shuffle: 2,289


In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────────────────
_EMPTY_STORE = {
    'store_count':       0,
    'active_licenses':   0,
    'outdated_licenses': 0,
    'type_counts_json':  '{}',
}

def reducer(zone_id, records):
    store_rec  = next((r for r in records if r['src'] == 'store'), None)
    crash_recs = [r for r in records if r['src'] == 'crash']
    if not crash_recs:
        return []
    sr = store_rec if store_rec else _EMPTY_STORE
    return [
        {
            'zone_id':           zone_id,
            'crash_date':        c['crash_date'],
            'crashes':           c['crashes'],
            'injured':           c['injured'],
            'killed':            c['killed'],
            'store_count':       sr['store_count'],
            'active_licenses':   sr['active_licenses'],
            'outdated_licenses': sr['outdated_licenses'],
            'type_counts_json':  sr['type_counts_json'],
        }
        for c in crash_recs
    ]

rows = []
for zone_id, records in shuffled.items():
    rows.extend(reducer(zone_id, records))

out = pd.DataFrame(rows).sort_values(['zone_id','crash_date']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

crash_only_zones = out[out['store_count'] == 0]['zone_id'].nunique()
print(f'Output rows      : {len(out):,}')
print(f'Unique zones     : {out["zone_id"].nunique():,}')
print(f'Crash-only zones : {crash_only_zones:,}  (store_count=0, retained via left join)')
print(f'Saved → {OUTPUT}')
out.head()

Output rows      : 2,891
Unique zones     : 1,490
Crash-only zones : 466  (store_count=0, retained via left join)
Saved → mr_daily_final.csv


,zone_id,crash_date,crashes,injured,killed,store_count,active_licenses,outdated_licenses,type_counts_json
0,8a2a1008800ffff,2017-02-11,1,2.0,0.0,0,0,0,{}
1,8a2a1008800ffff,2019-02-15,1,1.0,0.0,0,0,0,{}
2,8a2a1008802ffff,2020-11-05,1,0.0,0.0,0,0,0,{}
3,8a2a1008802ffff,2022-10-28,1,1.0,0.0,0,0,0,{}
4,8a2a10088047fff,2015-04-01,1,0.0,0.0,0,0,0,{}
